# Cox Proportional Hazards

### Wstęp
&emsp;Model proporcjonalnego hazardu Cox'a jest modelem pozwalającym ocenić wpływ wielu czynników na funkcję hazardu (chwilowe natężenie ryzyka). W naszym przypadku oceniać będziemy wpływ na ryzyko zapłaty (zdarzenie).

&emsp; Przy budownie modelu uwzględnimy następujące cechy:
* `total_open_amount` *(kwota faktury)*
* `days_to_due` *(po ilu dniach płatność)*
* `invoice_age` *(wiek faktury)*
* `avg_delay_customer` *(średnie opóźnienie klienta)*
* `cust_payment_terms` *(warunki płatności)*
* `invoice_currency` *(waluta)*
* `segment` *(BE lub SME)*

### Biblioteki i przygotowanie danych
&emsp; Zacznijmy od zaimportowania niezbędnych narzędzi oraz przygotowania danych. Zostawimy jedynie kolumny odpowiadające interesującym nas cechom.

In [20]:
import pandas as pd
from lifelines import CoxPHFitter
import matplotlib.pyplot as plt

df = pd.read_csv('../data/dataset_survclean.csv')

df['document_create_date_dt'] = pd.to_datetime(df['document_create_date_dt'])
df = df.sort_values('document_create_date_dt')

features = [
    'total_open_amount',
    'days_to_due',
    'invoice_age',
    'avg_delay_customer',
    'cust_payment_terms',
    'invoice_currency',
    'segment',
    'time_days', 
    'event'
]

cox_df = df[features].copy()

cox_df = pd.get_dummies(cox_df, columns=['cust_payment_terms', 'invoice_currency', 'segment'], drop_first=True)
cox_df = cox_df.dropna()
cox_df = cox_df.astype(float)

splt_id = int(len(cox_df) * 0.8)
train_df = cox_df.iloc[:splt_id]
test_df = cox_df.iloc[splt_id:]

print(f"Zbiór treningowy: {len(train_df)} faktur\nZbiór testowy: {len(test_df)} faktur")

display(train_df.head())

Zbiór treningowy: 39070 faktur
Zbiór testowy: 9768 faktur


,total_open_amount,days_to_due,invoice_age,avg_delay_customer,time_days,event,cust_payment_terms_NA10,cust_payment_terms_NA32,cust_payment_terms_NAA8,cust_payment_terms_NAAW,...,cust_payment_terms_NAVF,cust_payment_terms_NAVQ,cust_payment_terms_NAVR,cust_payment_terms_NAWN,cust_payment_terms_NAWP,cust_payment_terms_NAWU,cust_payment_terms_NAX2,cust_payment_terms_OTHER,invoice_currency_USD,segment_SME
0,13760.55,34.0,30.0,0.0,28.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
1,28225.48,32.0,30.0,0.0,62.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
53,4445.93,16.0,15.0,0.2,11.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
52,51473.49,16.0,15.0,0.0,17.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
51,4678.12,20.0,15.0,0.0,19.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


### Model
&emsp; Pozostało dopasować model Coxa do badanego zbioru danych. Kolumną zawierającą informacje na temat upływu czasu jest `time_days`, a kolumną informującą o wystąpieniu zdarzenia jest kolumna `event`.

In [21]:
cph = CoxPHFitter()

cph.fit(train_df, duration_col='time_days', event_col='event')

<lifelines.CoxPHFitter: fitted with 39070 total observations, 38 right-censored observations>

#### Załozenie o proporcjinalności hazardów
&emsp; Model Coxa opiera się na załozeniu o proporcjonalności hazardów. Załozenie to mowi, ze stosunek hazardu dla dowolnych dwoch obserwacji musi być stały w czasie. W kontekście projektu oznacza to, ze wpływ cechy na prawdopodobieństwo zapłacenia faktury nie moze się zmieniać wraz upływem dni. Wpływ ten musi być ponadto liniowy.

&emsp; Jeśli to załozenie jest łamane model traci na wiarygodności. W celu jego weryfikacji wykorzystamy test reszt Schoenfelda.

In [22]:
results = cph.check_assumptions(train_df, p_value_threshold=0.05, show_plots=False)

The ``p_value_threshold`` is set at 0.05. Even under the null hypothesis of no violations, some
covariates will be below the threshold by chance. This is compounded when there are many covariates.
Similarly, when there are lots of observations, even minor deviances from the proportional hazard
assumption will be flagged.

With that in mind, it's best to use a combination of statistical tests and visual tests to determine
the most serious violations. Produce visual plots using ``check_assumptions(..., show_plots=True)``
and looking for non-constant lines. See link [A] below for a full example.





1. Variable 'total_open_amount' failed the non-proportional test: p-value is <5e-05.

   Advice 1: the functional form of the variable 'total_open_amount' might be incorrect. That is,
there may be non-linear terms missing. The proportional hazard test used is very sensitive to
incorrect functional forms. See documentation in link [D] below on how to specify a functional form.

   Advice 2: try binning the variable 'total_open_amount' using pd.cut, and then specify it in
`strata=['total_open_amount', ...]` in the call in `.fit`. See documentation in link [B] below.

   Advice 3: try adding an interaction term with your time variable. See documentation in link [C]
below.


2. Variable 'days_to_due' failed the non-proportional test: p-value is <5e-05.

   Advice 1: the functional form of the variable 'days_to_due' might be incorrect. That is, there
may be non-linear terms missing. The proportional hazard test used is very sensitive to incorrect
functional forms. See documentation in lin

&emsp;Z tabeli odczytujemy, że dla wielu zmiennych (m.in. `avg_delay_customer`, `days_to_due`, `invoice_age`, `segment_SME`, `total_open_amount`), wartość p-value jest niższa niż próg 0.05. Oznacza to, że hipoteza zerowa testu Schoenfelda jest fałszywa. Co za tym idzie, wpływ cech na "ryzyko" opłacenia faktury nie jest stały w czasie.

&emsp;Mamy do czynienia ze złamaniem podstawowego założenia modelu Coxa, co pozwala wyciągnąć wniosek, że modele nieliniowe jak Random Survival Forest poradzą sobie z predykcją zauważalnie lepiej.

#### Ewaluacja modelu
&emsp; Bazując na budowie algorytmu do wyliczenia funkcji częściowej wiarygodności (partial likelihood), w której wzorze wspolczynnik dla kazdej cechy jest stały w czasie, otrzymane w tabeli wyniki mozemy interpretować jako uśredniowy wpływ poszczególnych cech na "ryzyko" zapłaty.

&emsp; Wywołanie `cph.print_summary` dostarczy nam informacji o m.in:
* Hazard Ratio (`exp(coef)`): Dla zmiennych kategorycznych stosunek hazardu między podmiotem posiadającym i nieposiadającym danej cechy. Dla zmiennych ciągłych zmiana intensywności spłaty przy wzroście cechy o jedną jednostkę. Interpretacja:
    * HR > 1: zwiększone ryzyko
    * HR = 1: brak wpływu
    * HR < 1: zmniejszone ryzyko
* C-index (`Concordance`): Odsetek par, które model poprawnie uporządkował pod względem czasu do wystąpienia zdarzenia.

In [25]:
cph.print_summary()
test_cid = cph.score(test_df, scoring_method="concordance_index")

print(f"test C-index: {test_cid: .4f}")

<lifelines.CoxPHFitter: fitted with 39070 total observations, 38 right-censored observations>
             duration col = 'time_days'
                event col = 'event'
      baseline estimation = breslow
   number of observations = 39070
number of events observed = 39032
   partial log-likelihood = -359928.34
         time fit was run = 2026-08-10 14:35:11 UTC

---
                          coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                 
total_open_amount         0.00      1.00      0.00            0.00            0.00                1.00                1.00
days_to_due              -0.06      0.94      0.00           -0.06           -0.06                0.94                0.94
invoice_age               0.03      1.04      0.00            0.03            0.04                1.03                1.04
avg_delay_customer       -0.04      0.96      0.00           -0.04           -0.04                0.96                0.96
cust_payment_terms_NA10   0.17      1.18      0.12           -0.07            0.40                0.93                1.50
cust_payment_terms_NA32  -0.68      0.51      0.12           -0.90           -0.45                0.41                0.64
cust_payment_terms_NAA8   0.03      1.03      0.11           -0.18            0.24                0.84                1.27
cust_payment_terms_NAAW   0.30      1.35      0.14            0.03            0.56                1.03                1.75
cust_payment_terms_NAAX   0.26      1.30      0.11            0.05            0.48                1.05                1.62
cust_payment_terms_NAC6   0.18      1.20      0.11           -0.04            0.39                0.96                1.48
cust_payment_terms_NAD1  -0.49      0.61      0.11           -0.70           -0.27                0.49                0.77
cust_payment_terms_NAD5  -0.81      0.44      0.12           -1.06           -0.57                0.35                0.56
cust_payment_terms_NAG2  -0.78      0.46      0.11           -1.00           -0.57                0.37                0.57
cust_payment_terms_NAGD  -0.42      0.65      0.14           -0.70           -0.15                0.50                0.86
cust_payment_terms_NAH4   0.73      2.08      0.11            0.52            0.94                1.69                2.56
cust_payment_terms_NAM1   5.02    150.87      0.13            4.75            5.28              116.07              196.11
cust_payment_terms_NAM2   4.98    145.44      0.12            4.74            5.22              114.44              184.84
cust_payment_terms_NAM4   2.61     13.63      0.11            2.39            2.83               10.96               16.94
cust_payment_terms_NAU5   0.17      1.18      0.11           -0.06            0.39                0.95                1.47
cust_payment_terms_NAVE  -0.68      0.51      0.13           -0.93           -0.43                0.39                0.65
cust_payment_terms_NAVF  -0.60      0.55      0.14           -0.87           -0.33                0.42                0.72
cust_payment_terms_NAVQ  -0.64      0.53      0.17           -0.97           -0.31                0.38                0.73
cust_payment_terms_NAVR  -0.15      0.86      0.20           -0.54            0.24                0.58                1.27
cust_payment_terms_NAWN  -0.54      0.58      0.18           -0.90           -0.19                0.41                0.83
cust_payment_terms_NAWP  -0.87      0.42      0.17           -1.20           -0.53                0.30                0.59
cust_payment_terms_NAWU  -0.69      0.50      0.15           -0.97           -0.40                0.38                0.67
cust_payment_terms_NAX2  -0.48      0.62      0.12           -0.73           -0.24                0.48                0.79
cust_payment_terms_OTHER -0.44      0.64      0.11           -0

test C-index:  0.4457
